# IT Helpdesk RAG System

**Nama:** Faraday Barr Fatahillah  

## Gambaran Umum

Proyek ini membangun sistem **Retrieval-Augmented Generation (RAG)** untuk IT Helpdesk menggunakan komponen lokal sepenuhnya (tanpa API berbayar):

| Komponen | Teknologi |
|---|---|
| Klasifikasi tiket | TF-IDF + Logistic Regression |
| Penyimpanan & pencarian vektor | ChromaDB (persisten) |
| Embedding teks | `all-MiniLM-L6-v2` (SentenceTransformer) |
| Generasi jawaban | Ollama (`llama3`) |
| Evaluasi kualitas | LLM-as-Judge (Groundedness / Coherence / Relevance) |

### Alur Kerja
```mermaid
flowchart TB
 subgraph SUMBER["Sumber Data"]
        DS["Ticket Data (Training Set)"]
        SOP["SOP IT Helpdesk"]
        DT["Ticket Data (Test Set)"]
  end
 subgraph PREP["Persiapan Offline"]
        TRAIN["Classification Model from Training Set (Category)<br>"]
        INDEX["Parsing Data dan Indexing"]
  end
 subgraph STORAGE["Penyimpanan"]
        PKL["Save Model"]
        CHROMA["Chroma DB"]
  end
 subgraph PIPELINE["Pipeline RAG"]
        C1["Classify Category (Filtering Questions)"]
        C2["Context Search"]
        C3["Generate Answer from Model"]
        C4["Evaluate Metrics<br>(Groundedness, Relevance, Coherence)<br>"]
  end
    DS --> TRAIN
    SOP --> INDEX
    DT --> C1
    TRAIN --> PKL
    INDEX --> CHROMA
    PKL --> C1
    CHROMA --> C2
    C1 --> C2
    C2 --> C3
    C3 --> C4
```

## Sel 0 – Konfigurasi Global

Mendefinisikan konstanta yang digunakan di seluruh notebook:

- **`OLLAMA_MODEL`** – Nama model LLM lokal yang dijalankan melalui Ollama (default: `llama3`).
- **`EMBED_MODEL`** – Model SentenceTransformer untuk mengubah teks menjadi vektor embedding.
- **`CHROMA_PATH`** – Direktori lokal tempat ChromaDB menyimpan data vektor secara persisten.
- **`COLLECTION`** – Nama koleksi di dalam ChromaDB yang menyimpan potongan teks SOP.
- **`VALID_CATEGORIES`** – Himpunan kategori tiket yang dikenali oleh sistem (`Access`, `Network`, `Hardware`, `Software`).

In [1]:
OLLAMA_MODEL   = "llama3"
EMBED_MODEL    = "all-MiniLM-L6-v2"
CHROMA_PATH    = "./chroma_db"
COLLECTION     = "helpdesk_tickets"

VALID_CATEGORIES = {"Access", "Network", "Hardware", "Software"}

## Sel 1 – Pelatihan Klasifikasi Tiket (TF-IDF + Logistic Regression)

### Tujuan
Melatih model klasifikasi teks yang akan menentukan kategori (Access / Network / Hardware / Software) dari setiap tiket IT Helpdesk secara otomatis.

### Langkah-Langkah
1. **Memuat data latih** dari `tickets_IT_helpdesk_150each.csv` (150 tiket per kategori) dan **data uji** dari `tickets_IT_helpdesk_testset_30each.csv` (30 tiket per kategori).
2. **Deduplikasi** – membuang baris dari data latih yang isunya juga ada di data uji untuk mencegah kebocoran data.
3. **Vektorisasi** – menggunakan `TfidfVectorizer` dengan `ngram_range=(1, 2)` agar fitur unigram dan bigram ditangkap.
4. **Pelatihan** – `LogisticRegression(max_iter=1000)` dipilih karena sederhana, cepat, dan cukup kuat untuk klasifikasi teks.
5. **Evaluasi** – mencetak `classification_report` (precision, recall, F1) pada data uji.
6. **Menyimpan model** – pasangan `(vectorizer, clf)` diserialisasi ke `classifier.pkl` menggunakan `pickle` agar dapat dimuat ulang tanpa melatih ulang.

### File yang Dibutuhkan
- `tickets_IT_helpdesk_150each.csv` – dataset pelatihan
- `tickets_IT_helpdesk_testset_30each.csv` – dataset pengujian

### Output
- `classifier.pkl` – model klasifikasi yang telah dilatih


In [2]:
import pandas as pd, pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df_train = pd.read_csv("tickets_IT_helpdesk_150each.csv")

df_test = pd.read_csv("tickets_IT_helpdesk_testset_30each.csv")

df_train = df_train[~df_train["issue"].isin(df_test["issue"])]

valid_cats = df_train["category"].unique()
df_test = df_test[df_test["category"].isin(valid_cats)]

X_train = df_train["issue"]
y_train = df_train["category"]

X_test = df_test["issue"]
y_test = df_test["category"]

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

print(classification_report(y_test, clf.predict(X_test_vec)))

with open("classifier.pkl", "wb") as f:
    pickle.dump((vectorizer, clf), f)

              precision    recall  f1-score   support

      Access       0.96      0.73      0.83        30
         ERP       0.96      0.83      0.89        30
    Hardware       1.00      0.57      0.72        30
     Network       0.72      0.97      0.83        30
       Other       0.91      1.00      0.95        30
    Software       0.66      0.90      0.76        30

    accuracy                           0.83       180
   macro avg       0.87      0.83      0.83       180
weighted avg       0.87      0.83      0.83       180



## Sel 2 – Parsing SOP dan Pengindeksan ke ChromaDB

### Tujuan
Mengurai dokumen SOP (Standard Operating Procedure) IT Helpdesk menjadi potongan-potongan teks (*chunks*) yang kemudian diindeks ke ChromaDB sebagai basis pengetahuan untuk retrieval.

### Penjelasan Fungsi `parse_sop(path)`
- Membaca file Markdown SOP baris per baris.
- Setiap heading `## SOP-00X` menandai awal bagian SOP baru dan menentukan kategori melalui `CATEGORY_MAP`.
- Setiap heading `### NNN.N` menandai satu prosedur dan menjadi satu *chunk* tersendiri.
- Setiap chunk menyimpan: `id`, `title`, `text` (isi prosedur), `sop`, dan `category`.

### Konfigurasi ChromaDB
- **`PersistentClient`** – data tersimpan di disk (`./chroma_db`) sehingga tidak perlu diindeks ulang setiap sesi.
- **`SentenceTransformerEmbeddingFunction`** – mengubah teks menjadi vektor menggunakan model `all-MiniLM-L6-v2` secara lokal.
- **Metrik jarak `cosine`** – mengukur kesamaan semantik antar-teks.

### File yang Dibutuhkan
- `SOP_IT_Helpdesk.md` – dokumen SOP dalam format Markdown

### Output
- Koleksi `helpdesk_tickets` di ChromaDB yang berisi semua potongan SOP beserta metadata-nya.


In [ ]:
import chromadb, re
from chromadb.utils import embedding_functions

emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL
)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION,
    embedding_function=emb_fn,
    metadata={"hnsw:space": "cosine"}
)


CATEGORY_MAP = {
    "SOP-001": "Access",
    "SOP-002": "Network",
    "SOP-003": "Hardware",
    "SOP-004": "Software",
    "SOP-005": "Software",
    "SOP-006": "Other",
}

def parse_sop(path: str) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        text = f.read()

    chunks = []
    current_sop = None
    current_title = None
    current_lines = []

    for line in text.splitlines():
        m_sop = re.match(r"^## (SOP-\d+)", line)
        if m_sop:
            current_sop = m_sop.group(1)

        m_proc = re.match(r"^### (\d{3}\.\d+)", line)
        if m_proc:
            if current_title and current_lines:
                chunks.append({
                    "id":       current_title,
                    "title":    current_title,
                    "text":     "\n".join(current_lines).strip(),
                    "sop":      current_sop,
                    "category": CATEGORY_MAP.get(current_sop, "Other"),
                })
            current_title = line.lstrip("# ").strip()
            current_lines = [line]
        elif current_title:
            current_lines.append(line)

    if current_title and current_lines:
        chunks.append({
            "id":       current_title,
            "title":    current_title,
            "text":     "\n".join(current_lines).strip(),
            "sop":      current_sop,
            "category": CATEGORY_MAP.get(current_sop, "Other"),
        })

    return chunks

SOP_PATH = "SOP_IT_Helpdesk.md"
sop_chunks = parse_sop(SOP_PATH)

print(f"Parsed {len(sop_chunks)} SOP sections:")
for c in sop_chunks:
    print(f"  [{c['category']:8s}] {c['title']}")

collection.upsert(
    ids       = [c["id"] for c in sop_chunks],
    documents = [c["text"] for c in sop_chunks],
    metadatas = [{"category": c["category"], "sop": c["sop"], "title": c["title"]} for c in sop_chunks],
)

print(f"\nIndexed {collection.count()} SOP chunks into ChromaDB")


c:\Users\bobe\anaconda3\envs\miniproject-5\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8029.09it/s]


Parsed 12 SOP sections:
  [Access  ] 001.1 · Reset Password Domain Windows
  [Network ] 001.2 · Reset Token MFA (Multi-Factor Authentication)
  [Network ] 002.1 · Troubleshooting GlobalProtect VPN
  [Hardware] 002.2 · Akses Wi-Fi Acme-Guest (Tamu & Vendor)
  [Hardware] 003.1 · Pengajuan Laptop Pengganti (Device Refresh)
  [Software] 003.2 · Permintaan Peripheral Tambahan
  [Software] 004.1 · Error SAP GUI: WSAECONNREFUSED
  [Software] 004.2 · Akun SAP Terkunci (Account Locked)
  [Software] 005.1 · Mailbox Outlook Penuh (Quota Exceeded)
  [Other   ] 005.2 · Permintaan Lisensi Microsoft Visio / Project
  [Other   ] 006.1 · Pemesanan Ruang Rapat & Konsumsi
  [Other   ] 006.2 · Penggantian ID Card Hilang / Rusak

Indexed 12 SOP chunks into ChromaDB


## Sel 3 – Memuat Ulang Model Klasifikasi

Memuat pasangan `(vectorizer, clf)` yang sebelumnya disimpan di `classifier.pkl`.
Sel ini diperlukan jika notebook dijalankan ulang tanpa menjalankan Sel 1, sehingga model tidak perlu dilatih dari awal.


In [4]:
import pickle, ollama

with open("classifier.pkl", "rb") as f:
    vectorizer, clf = pickle.load(f)

## Sel 4 – Fungsi `classify_query`

### Tujuan
Mengklasifikasikan teks pertanyaan/tiket ke salah satu kategori: `Access`, `Network`, `Hardware`, atau `Software`.

### Cara Kerja
1. Teks query divektorisasi menggunakan `vectorizer` yang telah dilatih.
2. Model `clf` menghitung probabilitas untuk setiap kelas.
3. Jika probabilitas tertinggi **di bawah 0.40**, fungsi mengembalikan `"Unknown"` – artinya model tidak cukup yakin dan query tidak akan diproses lebih lanjut.
4. Jika probabilitas cukup tinggi, kelas dengan nilai tertinggi dikembalikan sebagai kategori.

### Parameter
| Parameter | Tipe | Keterangan |
|---|---|---|
| `query` | `str` | Teks pertanyaan atau deskripsi masalah pengguna |

### Return
- `str` – salah satu dari `Access`, `Network`, `Hardware`, `Software`, atau `Unknown`.


In [5]:
def classify_query(query: str) -> str:
    """
    Returns one of: Access | Network | Hardware | Software
    or 'Unknown' when the model confidence is too low.
    """
    vec   = vectorizer.transform([query])
    proba = clf.predict_proba(vec)[0]
    best  = proba.max()

    if best < 0.40:
        return "Unknown"

    return clf.predict(vec)[0]

## Sel 5 – Fungsi `retrieve`

### Tujuan
Mengambil potongan SOP yang paling relevan dari ChromaDB berdasarkan kemiripan semantik dengan query pengguna, dibatasi pada kategori yang sama.

### Cara Kerja
1. Melakukan query ke koleksi ChromaDB dengan filter `{"category": category}` agar hasil retrieval tetap dalam kategori yang tepat.
2. ChromaDB menghitung jarak cosine antara embedding query dan embedding setiap chunk SOP.
3. Mengembalikan `top_k` (default: 3) chunk terdekat beserta metadata-nya.

### Parameter
| Parameter | Tipe | Default | Keterangan |
|---|---|---|---|
| `query` | `str` | – | Teks pertanyaan pengguna |
| `category` | `str` | – | Kategori tiket yang telah diklasifikasikan |
| `top_k` | `int` | `3` | Jumlah chunk SOP teratas yang diambil |

### Return
- `list[dict]` – daftar dictionary dengan kunci `issue` (teks chunk) dan `category`.


In [6]:
def retrieve(query: str, category: str, top_k: int = 3) -> list[dict]:
    """
    Fetches the top-k most similar tickets from the same category.
    Returns a list of {issue, category} dicts.
    """
    results = collection.query(
        query_texts = [query],
        n_results   = top_k,
        where       = {"category": category},
    )
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    return [{"issue": d, "category": m["category"]} for d, m in zip(docs, metas)]


## Sel 6 – Fungsi `response_generate`

### Tujuan
Fungsi inti pipeline RAG: menggabungkan hasil retrieval SOP dengan LLM (Ollama llama3) untuk menghasilkan jawaban yang informatif dan relevan.

### Cara Kerja
1. Memvalidasi apakah `category` termasuk dalam `VALID_CATEGORIES`. Jika tidak, langsung mengembalikan pesan default.
2. Memanggil fungsi `retrieve()` untuk mendapatkan 3 chunk SOP paling relevan.
3. Menyusun **system prompt** yang menyertakan chunk-chunk tersebut sebagai konteks referensi untuk LLM.
4. Mengirim prompt ke Ollama dengan parameter `temperature=0.3` (jawaban konsisten, tidak terlalu kreatif) dan `num_predict=512` (batas panjang jawaban).
5. Mengembalikan dictionary berisi `query`, `response` (jawaban LLM), dan `context` (teks referensi yang digunakan).

### Parameter
| Parameter | Tipe | Keterangan |
|---|---|---|
| `query` | `str` | Pertanyaan atau deskripsi masalah pengguna |
| `category` | `str` | Kategori yang telah diklasifikasikan |

### Return
- `dict` dengan kunci: `query`, `response`, `context`.


In [7]:
def response_generate(query: str, category: str) -> dict:
    """
    Returns {query, response, context}.
    Replaces AzureOpenAI + Azure AI Search with Ollama + ChromaDB.
    """
    if category not in VALID_CATEGORIES:
        return {"query": query, "response": "I don't have that information.", "context": ""}

    hits = retrieve(query, category)
    context_text = "\n".join(
        f"- [{h['category']}] {h['issue']}" for h in hits
    )

    system_prompt = (
        "You are a helpful IT helpdesk agent. "
        "Use the reference tickets below to answer the user's question. "
        "Be concise and actionable.\n\n"
        "== Similar past tickets ==\n"
        f"{context_text}\n"
        "== End of reference tickets =="
    )

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": query},
        ],
        options={"temperature": 0.3, "num_predict": 512},
    )

    answer = response["message"]["content"].strip()
    return {"query": query, "response": answer, "context": context_text}


print(classify_query("Bagaimana cara mengaktifkan MFA?"))
print(classify_query("Cara membuat nasi goreng"))    

Access
Unknown


## Sel 7 – Inferensi Batch pada Data Uji

### Tujuan
Menjalankan pipeline RAG lengkap (klasifikasi → retrieval → generasi) untuk seluruh tiket pada dataset uji dan menyimpan hasilnya.

### Cara Kerja
1. Iterasi setiap baris di `df_test`.
2. Untuk setiap tiket, `classify_query()` menentukan kategorinya.
3. Tiket dengan kategori `Unknown` dilewati (tidak diproses).
4. `response_generate()` menghasilkan jawaban menggunakan LLM dengan konteks SOP.
5. Hasil disimpan ke file `results.jsonl` menggunakan library `jsonlines` (format JSON Lines – satu objek per baris).

### Output
- `results.jsonl` – file JSON Lines berisi setiap record `{query, response, context}`.


In [8]:
import jsonlines

results = []

for _, row in df_test.iterrows():
    query = row["issue"]
    category = classify_query(query)

    if category not in VALID_CATEGORIES:
        continue

    result = response_generate(query, category)
    results.append(result)
    print(f"Done: {row['ticket-id']} | category: {category}")

with jsonlines.open("results.jsonl", mode="w") as writer:
    writer.write_all(results)

print(f"\nSaved {len(results)} records to results.jsonl")

Done: TKT-TEST-10001 | category: Access
Done: TKT-TEST-10002 | category: Access
Done: TKT-TEST-10003 | category: Access
Done: TKT-TEST-10004 | category: Access
Done: TKT-TEST-10008 | category: Access
Done: TKT-TEST-10009 | category: Access
Done: TKT-TEST-10011 | category: Software
Done: TKT-TEST-10012 | category: Access
Done: TKT-TEST-10013 | category: Access
Done: TKT-TEST-10014 | category: Access
Done: TKT-TEST-10016 | category: Access
Done: TKT-TEST-10017 | category: Access
Done: TKT-TEST-10019 | category: Access
Done: TKT-TEST-10020 | category: Access
Done: TKT-TEST-10022 | category: Access
Done: TKT-TEST-10023 | category: Access
Done: TKT-TEST-10029 | category: Access
Done: TKT-TEST-10031 | category: Network
Done: TKT-TEST-10034 | category: Network
Done: TKT-TEST-10035 | category: Network
Done: TKT-TEST-10036 | category: Network
Done: TKT-TEST-10037 | category: Network
Done: TKT-TEST-10039 | category: Network
Done: TKT-TEST-10040 | category: Network
Done: TKT-TEST-10042 | category

## Sel 8 – Fungsi Evaluasi LLM-as-Judge

### Tujuan
Mengevaluasi kualitas jawaban yang dihasilkan pipeline RAG menggunakan LLM sebagai juri (*LLM-as-Judge*) dengan tiga metrik:

| Metrik | Definisi |
|---|---|
| **Groundedness** | Apakah setiap klaim dalam jawaban didukung oleh konteks yang diberikan? |
| **Coherence** | Apakah jawaban terstruktur dengan logis, lancar, dan mudah diikuti? |
| **Relevance** | Apakah jawaban secara langsung menjawab pertanyaan pengguna? |

Setiap metrik menggunakan skala **1–5**.

### Penjelasan Fungsi

#### `_parse_score(raw: str) -> dict`
Mengurai respons mentah LLM (yang mungkin mengandung tanda kutip tidak standar atau pagar Markdown) menjadi dictionary `{"score": int, "reason": str}`. Dilengkapi fallback regex jika `json.loads` gagal.

#### `evaluate_single(record: dict) -> dict`
Menjalankan ketiga prompt evaluasi terhadap satu record hasil RAG. Mengembalikan record asli yang diperkaya dengan enam kolom baru:
- `eval_groundedness`, `eval_groundedness_reason`
- `eval_coherence`, `eval_coherence_reason`
- `eval_relevance`, `eval_relevance_reason`


In [ ]:
import ollama, json, re

EVAL_PROMPTS = {
    "groundedness": """
You are an evaluation assistant. Rate the GROUNDEDNESS of the response below.

Groundedness measures whether every claim in the response is directly supported
by the provided context. Ignore whether the answer is correct in the real world;
only judge if it is supported by the context.

Scale:
1 - Response contradicts or ignores the context entirely.
2 - Most claims are unsupported or contradict the context.
3 - Some claims are supported; others are not grounded in the context.
4 - Almost all claims are supported by the context.
5 - Every claim is fully and explicitly supported by the context.

Context:
{context}

Response:
{response}

Reply with ONLY a JSON object: {{"score": <1-5>, "reason": "<one sentence>"}}
""",

    "coherence": """
You are an evaluation assistant. Rate the COHERENCE of the response below.

Coherence measures whether the response is logically structured, fluent,
and easy to follow — independent of factual accuracy.

Scale:
1 - Incomprehensible or completely disjointed.
2 - Hard to follow; major structural problems.
3 - Understandable but somewhat disjointed or repetitive.
4 - Clear and well-structured with minor issues.
5 - Perfectly clear, logical, and easy to follow.

Response:
{response}

Reply with ONLY a JSON object: {{"score": <1-5>, "reason": "<one sentence>"}}
""",

    "relevance": """
You are an evaluation assistant. Rate the RELEVANCE of the response to the query.

Relevance measures whether the response directly addresses what the user asked.

Scale:
1 - Completely off-topic.
2 - Barely related; misses the main point.
3 - Partially addresses the query but misses key aspects.
4 - Mostly addresses the query with minor gaps.
5 - Fully and directly answers the query.

Query:
{query}

Response:
{response}

Reply with ONLY a JSON object: {{"score": <1-5>, "reason": "<one sentence>"}}
"""
}

def _parse_score(raw: str) -> dict:
    """Extract {score, reason} from the judge's raw output."""
    clean = re.sub(r"```[\w]*", "", raw).strip()
    clean = clean.replace("\u201c", "\"").replace("\u201d", "\"")
    clean = clean.replace("\u2018", "'").replace("\u2019", "'")
    m_obj = re.search(r"\{[^}]+\}", clean, re.DOTALL)
    if m_obj:
        clean = m_obj.group(0)
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        m = re.search(r"\b([1-5])\b", raw)
        score = int(m.group(1)) if m else 0
        r = re.search(r'reason["\':\\s]+([^,}]+)', raw, re.IGNORECASE)
        reason = r.group(1).strip().strip('"\' ') if r else raw[:120]
        return {"score": score, "reason": reason}


def evaluate_single(record: dict) -> dict:
    """
    Runs all three judges on one result record.
    record must have keys: query, response, context
    Returns the record enriched with eval_groundedness, eval_coherence, eval_relevance.
    """
    scores = {}
    for metric, prompt_template in EVAL_PROMPTS.items():
        prompt = prompt_template.format(
            query    = record.get("query", ""),
            response = record.get("response", ""),
            context  = record.get("context", "(no context)"),
        )
        raw = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0, "num_predict": 128},
        )["message"]["content"].strip()

        parsed = _parse_score(raw)
        scores[f"eval_{metric}"] = parsed.get("score", 0)
        scores[f"eval_{metric}_reason"] = parsed.get("reason", "")

    return {**record, **scores}


print("Evaluator ready.")

Evaluator ready.


## Sel 9 – Evaluasi Batch Seluruh Hasil RAG

### Tujuan
Memuat file `results.jsonl` dan menjalankan evaluasi LLM-as-Judge pada setiap record, kemudian menyimpan hasilnya.

### Cara Kerja
1. Membaca semua record dari `results.jsonl`.
2. Memanggil `evaluate_single()` untuk setiap record – setiap panggilan mengirimkan 3 request ke Ollama (satu per metrik).
3. Mencetak skor secara real-time: `G=` (Groundedness), `C=` (Coherence), `R=` (Relevance).
4. Menyimpan semua record yang telah dievaluasi ke `results_evaluated.jsonl`.

### Output
- `results_evaluated.jsonl` – record lengkap dengan skor dan alasan dari ketiga metrik evaluasi.


In [ ]:
import jsonlines

with jsonlines.open("results.jsonl") as reader:
    rag_results = list(reader)

print(f"Evaluating {len(rag_results)} records...")

evaluated = []
for i, record in enumerate(rag_results):
    scored = evaluate_single(record)
    evaluated.append(scored)
    print(
        f"[{i+1}/{len(rag_results)}] "
        f"G={scored['eval_groundedness']} "
        f"C={scored['eval_coherence']} "
        f"R={scored['eval_relevance']}  "
        f"| {record['query'][:60]}"
    )

with jsonlines.open("results_evaluated.jsonl", mode="w") as writer:
    writer.write_all(evaluated)

print(f"\nSaved to results_evaluated.jsonl")

Evaluating 61 records...
[1/61] G=4 C=4 R=5  | Karyawan tidak bisa login ke sistem manajemen dokumen ShareP
[2/61] G=5 C=4 R=5  | Akun VDI (Virtual Desktop Infrastructure) karyawan terkunci 
[3/61] G=4 C=4 R=4  | Karyawan tidak bisa mengakses portal pengajuan cuti online k
[4/61] G=5 C=4 R=5  | Login ke sistem monitoring CCTV jaringan ditolak meskipun su
[5/61] G=4 C=4 R=5  | Akun email karyawan dinonaktifkan secara otomatis karena tid
[6/61] G=4 C=4 R=5  | Karyawan tidak bisa login ke portal e-procurement karena aku
[7/61] G=4 C=4 R=5  | Karyawan tidak bisa akses aplikasi mobile ESS (Employee Self
[8/61] G=4 C=4 R=5  | Hak akses karyawan ke sistem approval anggaran belum dicabut
[9/61] G=2 C=4 R=5  | Login ke Zoom dengan akun korporat gagal karena karyawan men
[10/61] G=4 C=4 R=5  | Karyawan tidak bisa akses sistem helpdesk ITSM untuk membuka
[11/61] G=2 C=4 R=5  | Karyawan tidak bisa akses repositori Git internal karena bel
[12/61] G=2 C=4 R=5  | Login ke dashboard Power BI korporat 

## Sel 10 – Statistik Ringkasan Evaluasi

### Tujuan
Menampilkan ringkasan statistik agregat dari hasil evaluasi untuk menilai performa keseluruhan sistem RAG.

### Cara Kerja
1. Mengubah daftar record yang telah dievaluasi menjadi `DataFrame` pandas.
2. Menghitung statistik deskriptif (`mean`, `min`, `max`, `std`) untuk ketiga metrik evaluasi.
3. Mencetak rata-rata setiap metrik dalam format `X.XX / 5.00`.


In [11]:
import pandas as pd

eval_df = pd.DataFrame(evaluated)

metrics = ["eval_groundedness", "eval_coherence", "eval_relevance"]

summary = eval_df[metrics].agg(["mean", "min", "max", "std"]).round(2)
summary.columns = [m.replace("eval_", "").capitalize() for m in metrics]
print(summary.to_string())

print("\n── Overall averages ──")
for m in metrics:
    label = m.replace("eval_", "").capitalize()
    print(f"{label:15s}: {eval_df[m].mean():.2f} / 5.00")

      Groundedness  Coherence  Relevance
mean          3.90       4.02       4.69
min           2.00       4.00       2.00
max           5.00       5.00       5.00
std           0.51       0.13       0.62

── Overall averages ──
Groundedness   : 3.90 / 5.00
Coherence      : 4.02 / 5.00
Relevance      : 4.69 / 5.00


## Sel 11 – Identifikasi Record Bermasalah

### Tujuan
Menampilkan secara detail tiket-tiket yang mendapatkan skor rendah pada setidaknya satu metrik evaluasi, untuk keperluan analisis dan perbaikan sistem.

### Cara Kerja
1. Memfilter `eval_df` untuk mencari record di mana **Groundedness**, **Coherence**, atau **Relevance** di bawah ambang batas `THRESHOLD` (default: **3**).
2. Menampilkan query, 200 karakter pertama respons, ketiga skor, dan alasan dari juri untuk setiap record yang ditandai.

### Interpretasi
- **Skor Groundedness rendah** → jawaban LLM mengandung klaim yang tidak didukung konteks SOP; pertimbangkan untuk memperbaiki prompt atau menambah chunk SOP.
- **Skor Coherence rendah** → jawaban sulit diikuti; kemungkinan perlu pengaturan parameter generasi.
- **Skor Relevance rendah** → classifier salah mengkategorikan tiket atau konteks yang diambil kurang tepat.


In [ ]:
THRESHOLD = 3

flagged = eval_df[
    (eval_df["eval_groundedness"] < THRESHOLD) |
    (eval_df["eval_coherence"]    < THRESHOLD) |
    (eval_df["eval_relevance"]    < THRESHOLD)
]

print(f"{len(flagged)} records scored below {THRESHOLD} on at least one metric:\n")

for _, row in flagged.iterrows():
    print(f"Query   : {row['query']}")
    print(f"Response: {row['response'][:200]}")
    print(f"Scores  : G={row['eval_groundedness']}  C={row['eval_coherence']}  R={row['eval_relevance']}")
    print(f"Reasons : {row['eval_groundedness_reason']}")
    print("-" * 80)

4 records scored below 3 on at least one metric:

Query   : Login ke Zoom dengan akun korporat gagal karena karyawan mencoba menggunakan akun Zoom personal
Response: I'd be happy to help you with that!

It looks like the issue is due to a common mistake where employees try to log in to Zoom using their personal account instead of their corporate account. To resolv
Scores  : G=2  C=4  R=5
Reasons : The response contradicts the context by suggesting that the issue is related to Zoom and corporate email addresses, whereas the original context is about resetting passwords for domain Windows accounts.
--------------------------------------------------------------------------------
Query   : Karyawan tidak bisa akses repositori Git internal karena belum ditambahkan ke project yang relevan
Response: To help the employee access the internal Git repository, I'll guide them through the process. Please follow these steps:

1. Log in to our company's portal and navigate to the "Projects" section.
